#### [Notebook 2 Quick Start](https://confluence.shopee.io/x/7O0lhw)
#### Common Use Cases and Solution Using Notebook 2
- [Multi Kernels in Notebook 2](https://confluence.shopee.io/x/4Od2hw)
- [Execute HDFS command](https://confluence.shopee.io/x/P5w8hw)
- [Browse, upload and download from Data Infra S3](https://confluence.shopee.io/x/xo9Ohw)
- [Setting up 3rd party packages for your Notebook 2](https://confluence.shopee.io/x/JZFOhw)
- [Develop and setup user defined function or library](https://confluence.shopee.io/x/hpjehg)
- [Execute Presto/Trino query directly in Python Kernel Notebook](https://confluence.shopee.io/x/WpMlhw)

> Note: "Python on K8S" Kernel cannot execute Spark functions. ***If you want to use Spark, change kernel to Spark***. [How to change Kernel](https://confluence.shopee.io/x/4Od2hw#MultiKernelsinNotebook2-SetKernelwhencreatingnewNotebook2file)

In [ ]:
# --********************************************************************--
# -- Author:              zongyi.lu@shopee.com
# -- Created Time:        2024-06-07
# -- Theme Domain：       seller
# -- Description:         tw_seller信息传给BE(因为原来的同步工具无法传成xlsx格式,因此需要自己转成xlsx后再上传s3)
# -- Data Granularity:    risk.ads_risk_tw_seller_kyc_info_df
# -- 变更历史：
# --      update by 2024-06-07 zongyi.lu@shopee.com 1.新增注释
# --********************************************************************--



In [1]:
#!/usr/bin/python3
# encoding: utf-8
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
import pandas as pd
import re,json,sys,os
import random
import math
import datetime
import argparse


spark = SparkSession.builder \
            .appName('init_us_risk_table') \
            .master('yarn') \
            .config('spark.sql.execution.arrow.pyspark.enabled', True) \
            .config('spark.sql.session.timeZone', 'UTC') \
            .config('spark.driver.memory','16G') \
            .config('spark.executor.memory', '16g') \
            .config('spark.executor.cores', '4') \
            .config('spark.num.executors', '64') \
            .config('spark.ui.showConsoleProgress', True) \
            .config('hive.exec.dynamic.partition.mode', 'nonstrict') \
            .config('spark.sql.repl.eagerEval.enabled', True) \
            .enableHiveSupport() \
            .getOrCreate()

storage_options = {
    # The credentials are not required to be set if they are already set in the environment variables / credentials file.
    "key": "zongyi.lu",
    "secret": "4kgnyF27zxhh",
    # for SG idc, use this endpoint
    # for others can refer https://confluence.shopee.io/display/DI/3.+DI+S3+Usage
    "client_kwargs": {"endpoint_url": "https://s3g.data-infra.shopee.io"}
}

# 设置Hive数据库和表名
sql = "select store_id,brand_name as brand_nme,resp_name  as resp_nme,resp_id    as resp_id,vendor_name as vendor_nme,vendor_no   as vendor_no  from risk.ads_risk_tw_seller_kyc_info_df where grass_date ='${grass_date}'"
print(sql)
df = spark.sql(sql)
pandas_df = df.toPandas()
# TW/live/SHP_E/SHP_E_20240605.xlsx
pandas_df.to_excel("s3://risk-feature/TW/live/SHP_E/SHP_E_${BIZ_DATE}.xlsx", index=False,storage_options=storage_options)






select store_id,brand_name as brand_nme,resp_name  as resp_nme,resp_id    as resp_id,vendor_name as vendor_nme,vendor_no   as vendor_no  from risk.ads_risk_tw_seller_kyc_info_df where grass_date ='2024-06-12'